# Activation Probing

Rich activation probing tools: multi-class probing, nonlinear probes,
concept localization through probing, cross-layer probe transfer, and
probe selectivity analysis.

## Why This Matters

Probing activation tests what information is linearly accessible in model representations. Linear probes provide a principled way to measure whether specific concepts (syntax, semantics, factual knowledge) are encoded at each layer.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_probing import (
    multiclass_probe,
    nonlinear_probe,
    concept_localization,
    cross_layer_probe_transfer,
    probe_selectivity,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Create labeled dataset (different token ranges = different classes)
tokens_list = [
    jnp.array([1, 2, 3, 4, 5, 6, 7]),
    jnp.array([2, 3, 4, 5, 6, 7, 8]),
    jnp.array([50, 51, 52, 53, 54, 55, 56]),
    jnp.array([51, 52, 53, 54, 55, 56, 57]),
    jnp.array([1, 3, 5, 7, 9, 11, 13]),
    jnp.array([50, 52, 54, 56, 58, 60, 62]),
]
labels = [0, 0, 1, 1, 0, 1]
print("Model and dataset ready")

## Multi-Class Probe

In [ ]:
result = multiclass_probe(model, tokens_list, labels)
print(f"Accuracy: {result['accuracy']:.1%}")
print(f"Class accuracies: {result['class_accuracies']}")
print(f"\nMost discriminative dimensions:")
for dim, imp in result['most_discriminative_dims'][:5]:
    print(f"  dim {dim}: importance={imp:.6f}")
print(f"\nConfusion matrix:")
print(result['confusion_matrix'])

## Nonlinear Probe

In [ ]:
result = nonlinear_probe(model, tokens_list, labels)
print(f"Nonlinear accuracy: {result['accuracy']:.1%}")
print(f"Linear accuracy: {result['linear_accuracy']:.1%}")
print(f"Nonlinearity gain: {result['nonlinearity_gain']:+.1%}")

## Concept Localization

Find where the concept becomes linearly separable.

In [ ]:
result = concept_localization(model, tokens_list, labels)
print(f"Emergence layer: {result['emergence_layer']}")
print(f"Peak layer: {result['peak_layer']}")
for l, acc in enumerate(result['accuracy_trajectory']):
    bar = '#' * int(acc * 20)
    print(f"  Layer {l}: {acc:.1%} {bar}")

## Cross-Layer Probe Transfer

In [ ]:
for train_l in range(cfg.n_layers):
    result = cross_layer_probe_transfer(model, tokens_list, labels, train_layer=train_l)
    accs = [f'{a:.0%}' for a in result['transfer_accuracies']]
    print(f"Train L{train_l}: transfer={accs}, best_other=L{result['best_transfer_layer']}")

## Probe Selectivity

In [ ]:
result = probe_selectivity(model, tokens_list, labels)
print(f"Selectivity: {result['selectivity_score']:.4f}")
print(f"Noise ratio: {result['noise_ratio']:.4f}")
print(f"Dimensions used: {result['dimension_usage']}")
print(f"Class separations: {result['class_separations']}")